# Data Workflow: Wine Quality

**Author:** Vivek Kumar

**Dataset:** Wine Quality (Red + White) — chemical properties and quality scores, from the UCI Machine Learning Repository ([dataset link](https://archive.ics.uci.edu/dataset/186/wine+quality))

**Project description:** This project builds a complete, reproducible data workflow on the UCI Wine Quality dataset, combining the red and white wine samples into a single dataset distinguished by wine type. It loads, cleans, transforms, explores, and visualizes the physicochemical measurements to understand how they relate to expert quality scores. The goal is to establish a clean, well-documented analytical foundation that future machine learning and AI projects can build on.

## 1. Setup

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 2. Load Data

The dataset ships as two semicolon-separated CSV files (red and white). We load each, tag it with a `wine_type` column, and concatenate the two into a single DataFrame so both varieties can be analyzed together.

In [4]:
# Files use ';' as the separator
red = pd.read_csv('data/winequality-red.csv', sep=';')
white = pd.read_csv('data/winequality-white.csv', sep=';')

# Tag each row with its wine type before combining
red['wine_type'] = 'red'
white['wine_type'] = 'white'

# Combine into a single DataFrame
df = pd.concat([red, white], ignore_index=True)

print(f'Red:   {red.shape[0]} rows')
print(f'White: {white.shape[0]} rows')
print(f'Combined: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

Red:   1599 rows
White: 4898 rows
Combined: 6497 rows, 13 columns


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,wine_type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red


## 3. Clean Data

We define reusable cleaning functions, each with a docstring describing its purpose, and then apply them to the combined dataset. Writing the cleaning steps as functions keeps the workflow reproducible: the same transformations can be re-run on new data or re-used in later projects.

In [5]:
def clean_column_names(df):
    """Standardize DataFrame column names for consistent, code-friendly access.

    Each column name is lowercased, stripped of leading/trailing whitespace, and
    has internal spaces replaced with underscores (e.g. 'fixed acidity' ->
    'fixed_acidity'). This prevents subtle bugs caused by inconsistent naming and
    enables attribute-style column access.

    Args:
        df (pd.DataFrame): The DataFrame whose columns should be standardized.

    Returns:
        pd.DataFrame: A copy of the DataFrame with cleaned column names.
    """
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(' ', '_')
    )
    return df

In [6]:
def drop_duplicate_rows(df):
    """Remove exact duplicate rows from the DataFrame.

    The Wine Quality dataset contains many identical records (same chemical
    measurements and quality score), which can bias summary statistics and
    visualizations. This function drops fully duplicated rows, resets the index,
    and reports how many rows were removed.

    Args:
        df (pd.DataFrame): The DataFrame to de-duplicate.

    Returns:
        pd.DataFrame: A copy of the DataFrame with duplicate rows removed.
    """
    df = df.copy()
    before = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    removed = before - len(df)
    print(f'Removed {removed} duplicate rows ({before} -> {len(df)}).')
    return df

In [7]:
def report_missing_values(df):
    """Summarize missing values per column.

    Returns a tidy summary of how many values are missing in each column and the
    corresponding percentage, sorted from most to least missing. This is a
    diagnostic step that documents data completeness before analysis; the Wine
    Quality dataset is expected to be complete, and this function makes that
    assumption explicit and verifiable.

    Args:
        df (pd.DataFrame): The DataFrame to inspect.

    Returns:
        pd.DataFrame: A summary with 'missing_count' and 'missing_pct' columns.
    """
    missing = df.isnull().sum()
    summary = pd.DataFrame({
        'missing_count': missing,
        'missing_pct': (missing / len(df) * 100).round(2),
    })
    return summary.sort_values('missing_count', ascending=False)

### Apply the cleaning functions

In [8]:
# Apply cleaning functions in sequence
df = clean_column_names(df)
df = drop_duplicate_rows(df)

# Diagnostic: confirm there are no missing values
report_missing_values(df)

Removed 1177 duplicate rows (6497 -> 5320).


,missing_count,missing_pct
fixed_acidity,0,0.0
volatile_acidity,0,0.0
citric_acid,0,0.0
residual_sugar,0,0.0
chlorides,0,0.0
free_sulfur_dioxide,0,0.0
total_sulfur_dioxide,0,0.0
density,0,0.0
ph,0,0.0
sulphates,0,0.0


In [9]:
# Verify the result of cleaning
print(f'Cleaned dataset: {df.shape[0]} rows, {df.shape[1]} columns')
print('Columns:', list(df.columns))
df.head()

Cleaned dataset: 5320 rows, 13 columns
Columns: ['fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol', 'quality', 'wine_type']


,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,ph,sulphates,alcohol,quality,wine_type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,red
4,7.4,0.66,0.00,1.8,0.075,13.0,40.0,0.9978,3.51,0.56,9.4,5,red


## 4. Explore & Analyze

In [10]:
df.describe()

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,ph,sulphates,alcohol,quality
count,5320.000000,5320.000000,5320.000000,5320.000000,5320.000000,5320.000000,5320.000000,5320.000000,5320.000000,5320.000000,5320.000000,5320.000000
mean,7.215179,0.344130,0.318494,5.048477,0.056690,30.036654,114.109023,0.994535,3.224664,0.533357,10.549241,5.795677
std,1.319671,0.168248,0.147157,4.500180,0.036863,17.805045,56.774223,0.002966,0.160379,0.149743,1.185933,0.879772
min,3.800000,0.080000,0.000000,0.600000,0.009000,1.000000,6.000000,0.987110,2.720000,0.220000,8.000000,3.000000
25%,6.400000,0.230000,0.240000,1.800000,0.038000,16.000000,74.000000,0.992200,3.110000,0.430000,9.500000,5.000000
50%,7.000000,0.300000,0.310000,2.700000,0.047000,28.000000,116.000000,0.994650,3.210000,0.510000,10.400000,6.000000
75%,7.700000,0.410000,0.400000,7.500000,0.066000,41.000000,153.250000,0.996770,3.330000,0.600000,11.400000,6.000000
max,15.900000,1.580000,1.660000,65.800000,0.611000,289.000000,440.000000,1.038980,4.010000,2.000000,14.900000,9.000000
